In [1]:
import os
import sys
sys.path.append("..")
sys.path.append("../ALAE")

import torch
import numpy as np
from src.light_sb_ou import LightSB_OU
from src.distributions import TensorSampler
from alae_ffhq_inference import load_model, decode
from PIL import Image
from tqdm import tqdm
import time
import glob
import torch.nn as nn
from scipy import linalg
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import warnings
warnings.filterwarnings('ignore')

In [2]:
def setup_consistent_evaluation():
    EVAL_SEED = 42  
    torch.manual_seed(EVAL_SEED)
    np.random.seed(EVAL_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(EVAL_SEED)

In [3]:
setup_consistent_evaluation()

# Config

In [4]:
BASE_DIR = r"F:\data"
DIM = 512

fid_real_dir = os.path.join(BASE_DIR, "fid_real_1000")
fid_generated_dir = os.path.join(BASE_DIR, "fid_generated_1000")
os.makedirs(fid_real_dir, exist_ok=True)
os.makedirs(fid_generated_dir, exist_ok=True)

# Model Training

In [5]:
alae_model = load_model("../ALAE/configs/ffhq.yaml", training_artifacts_dir="../ALAE/training_artifacts/ffhq/")

In [6]:
latents = np.load("../data/latents.npy")
gender = np.load("../data/gender.npy")
age = np.load("../data/age.npy")
    
train_size = 60000
train_latents = latents[:train_size]
train_gender = gender[:train_size]
test_latents = latents[train_size:]
test_gender = gender[train_size:]
test_age = age[train_size:]
    
source_inds = np.arange(len(test_gender))[(test_gender == "male").reshape(-1)]
source_latents = test_latents[source_inds] 

target_inds = np.arange(len(test_gender))[(test_gender == "female").reshape(-1)]
target_latents = test_latents[target_inds]

In [7]:
print("Filtering MAN (source) and WOMAN (target) for training...")
man_inds = np.arange(train_size)[(train_gender == "male").reshape(-1)]
woman_inds = np.arange(train_size)[(train_gender == "female").reshape(-1)]
    
X_train = torch.tensor(train_latents[man_inds])   
Y_train = torch.tensor(train_latents[woman_inds])
    
print(f"Training samples - MAN: {len(X_train)}, WOMAN: {len(Y_train)}")
    
X_sampler = TensorSampler(X_train, device="cpu")
Y_sampler = TensorSampler(Y_train, device="cpu")
    
D = LightSB_OU(
    dim=DIM, 
    n_potentials=10, 
    epsilon=0.1, 
    b=0.02,
    m=0.0,
    sampling_batch_size=128, 
    S_diagonal_init=0.1,
    is_diagonal=True
).cpu()
    
D.init_r_by_samples(Y_sampler.sample(10))
D_opt = torch.optim.Adam(D.parameters(), lr=1e-3)
    
print("Starting training...")
MAX_STEPS = 5000 
    
for step in tqdm(range(MAX_STEPS), desc="Training"):
    D_opt.zero_grad()
    X0, X1 = X_sampler.sample(128), Y_sampler.sample(128)
        
    log_potential = D.get_log_potential(X1)
    log_C = D.get_log_C(X0)
        
    D_loss = (-log_potential + log_C).mean()
    D_loss.backward()
    torch.nn.utils.clip_grad_norm_(D.parameters(), max_norm=float("inf"))
    D_opt.step()
        
    if step % 1000 == 0:
        print(f"Step {step}: Loss = {D_loss.item():.6f}")
    
D.eval()

Filtering MAN (source) and WOMAN (target) for training...
Training samples - MAN: 26732, WOMAN: 32816
Starting training...


Training:   0%|                                                                       | 7/5000 [00:00<01:19, 63.06it/s]

Step 0: Loss = 18280.113281


Training:  20%|█████████████▋                                                     | 1020/5000 [00:09<00:37, 106.65it/s]

Step 1000: Loss = 6344.626953


...

Step 3000: Loss = 4204.577637


Training:  80%|█████████████████████████████████████████████████████▋             | 4009/5000 [00:37<00:09, 107.66it/s]

Step 4000: Loss = 4080.543945


Training: 100%|███████████████████████████████████████████████████████████████████| 5000/5000 [00:46<00:00, 107.58it/s]


ParametrizedLightSB_OU(
  (parametrizations): ModuleDict(
    (S_rotation_matrix): ParametrizationList(
      (0): Stiefel(n=512, k=512, tensorial_size=(10,), triv=linalg_matrix_exp)
    )
  )
)

# Generation

In [8]:
total_images = 1000
batch_size = 32

In [9]:
def decode_and_save_batch(model, latents, output_dir, start_idx, batch_size=8):
    with torch.no_grad():
        for i in range(0, len(latents), batch_size):
            end_idx = min(i + batch_size, len(latents))
            batch_latents = latents[i:end_idx]
            
            batch_images = decode(model, batch_latents)
            batch_images = (batch_images * 0.5 + 0.5).clamp(0, 1)
            
            for j, img in enumerate(batch_images):
                global_idx = start_idx + i + j
                img_np = (img.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
                pil_img = Image.fromarray(img_np)
                pil_img.save(os.path.join(output_dir, f"image_{global_idx:05d}.png"))
            
            del batch_images
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

In [10]:
print(f"\nGenerating {total_images} REAL images (WOMAN)...")
real_count = 0
for start_idx in tqdm(range(0, total_images, batch_size), desc="Real images"):
    end_idx = min(start_idx + batch_size, total_images)
    batch_latents = torch.tensor(target_latents[start_idx:end_idx])
    decode_and_save_batch(alae_model, batch_latents, fid_real_dir, start_idx, batch_size=8)
    real_count += (end_idx - start_idx)
        
    del batch_latents
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Generating 1000 REAL images (WOMAN)...


Real images: 100%|█████████████████████████████████████████████████████████████████████| 32/32 [22:52<00:00, 42.88s/it]


In [11]:
print(f"\nGenerating {total_images} GENERATED images (MAN -> WOMAN)...")
generated_count = 0
for start_idx in tqdm(range(0, total_images, batch_size), desc="Generated images"):
    end_idx = min(start_idx + batch_size, total_images)
        
    source_batch = torch.tensor(source_latents[start_idx:end_idx])
        
    with torch.no_grad():
        transformed_batch = D(source_batch.cpu())
        
    decode_and_save_batch(alae_model, transformed_batch, fid_generated_dir, start_idx, batch_size=8)
    generated_count += (end_idx - start_idx)
        
    del source_batch, transformed_batch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Generating 1000 GENERATED images (MAN -> WOMAN)...


Generated images: 100%|████████████████████████████████████████████████████████████████| 32/32 [23:05<00:00, 43.30s/it]


# FID calculation

In [12]:
class SimpleFIDCalculator:
    def __init__(self, device='cpu'):
        self.device = device
        self.transform = transforms.Compose([
            transforms.Resize(299),
            transforms.CenterCrop(299),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
    def load_inception_model(self):
        try:
            import torchvision.models as models
            
            try:
                model = models.inception_v3(weights='DEFAULT')
            except:
                model = models.inception_v3(pretrained=True)
            
            model.eval()
            model.fc = nn.Identity()
            return model.to(self.device)
            
        except Exception as e:
            print(f"Could not load Inception model: {e}")
            return None
    
    def extract_features(self, directory, batch_size=16):
        model = self.load_inception_model()
        if model is None:
            return None
            
        class ImageDataset(Dataset):
            def __init__(self, directory, transform):
                self.image_files = [os.path.join(directory, f) for f in os.listdir(directory) 
                                  if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
                self.transform = transform
            
            def __len__(self):
                return len(self.image_files)
            
            def __getitem__(self, idx):
                try:
                    image = Image.open(self.image_files[idx]).convert('RGB')
                    return self.transform(image)
                except Exception as e:
                    print(f"Error loading {self.image_files[idx]}: {e}")
                    return torch.zeros(3, 299, 299)
        
        dataset = ImageDataset(directory, self.transform)
        if len(dataset) == 0:
            print(f"No images found in {directory}")
            return None
            
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, 
                               num_workers=0, pin_memory=False)
        
        features = []
        with torch.no_grad():
            for batch in tqdm(dataloader, desc=f"Processing {os.path.basename(directory)}"):
                batch = batch.to(self.device)
                feat = model(batch)
                features.append(feat.cpu().numpy())
        
        return np.concatenate(features, axis=0)
    
    def calculate_fid(self, real_dir, gen_dir, batch_size=16):
        print("Extracting features from real images...")
        real_features = self.extract_features(real_dir, batch_size)
        if real_features is None:
            return None
            
        print("Extracting features from generated images...")
        gen_features = self.extract_features(gen_dir, batch_size)
        if gen_features is None:
            return None
        
        print("Calculating FID...")
        mu1, sigma1 = np.mean(real_features, axis=0), np.cov(real_features, rowvar=False)
        mu2, sigma2 = np.mean(gen_features, axis=0), np.cov(gen_features, rowvar=False)
        
        diff = mu1 - mu2
        covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
        
        if np.iscomplexobj(covmean):
            covmean = covmean.real
            
        fid = diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)
        return fid

In [13]:
print("=== Robust FID Calculation ===")
    
real_count = len([f for f in os.listdir(fid_real_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
gen_count = len([f for f in os.listdir(fid_generated_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        
calculator = SimpleFIDCalculator(device='cuda' if torch.cuda.is_available() else 'cpu')
fid_value = calculator.calculate_fid(fid_real_dir, fid_generated_dir, batch_size=8)  

print(f"\nFID = {fid_value:.6f}")

=== Robust FID Calculation ===
Extracting features from real images...


Processing fid_real_1000: 100%|██████████████████████████████████████████████████████| 125/125 [02:40<00:00,  1.28s/it]


Extracting features from generated images...


Processing fid_generated_1000: 100%|█████████████████████████████████████████████████| 125/125 [02:43<00:00,  1.31s/it]


Calculating FID...

FID = 24.019523
